# Axion PINN V14 — Extended Domain t ∈ [1e-8, 1]

## Design Overview

Extends V13 from 5 decades (t ∈ [1e-5, 1]) to **7 decades (t ∈ [1e-8, 1])** using a refined 8-stage curriculum.

### Sub-domain normalization
| Parameter | V13 | V14 |
|---|---|---|
| TAU_SPLIT | 0.50 | **0.20** |
| t_split | 1e-5 | **1e-8** |
| sub_log_dt | 11.513 | **18.421** |
| ds/dt @ t_split | 8 686 | **5 430 000** |

### Key insight
Sub-domain normalization `s = (ln t − ln t_split) / sub_log_dt ∈ [0,1]` maps exactly **one decade per 0.125 in s-space**:

| s_min | t_min | ds/dt | Stage |
|---|---|---|---|
| 0.875 | 1e-1 | 0.54 | A |
| 0.750 | 1e-2 | 5.4 | B |
| 0.625 | 1e-3 | 54 | C |
| 0.500 | 1e-4 | 543 | D |
| 0.375 | 1e-5 | 5 430 | E |
| 0.250 | 1e-6 | 54 300 | F |
| 0.125 | 1e-7 | 543 000 | G |
| 0.000 | 1e-8 | 5 430 000 | H |

Stages A–E overlap with V13 (A–E start from 0.875, V13 started from 0.80) giving a warm-start advantage.  
Stages F–H are fully new territory but the field is **completely frozen** (mₐt ≪ 1), making the physics trivial despite large ds/dt.

### Architecture
- Networks: `ScaleFactorNet`, `PhiNet (WKB)`, `DPhiNet (WKB)` — hidden=**128**, depth=5 (wider than V13 to cover 7 decades)
- WKB ICs at t_split=1e-8: mₐ·t_split = 1e-6 rad (deeply frozen, φ ≈ 1, φ̇ ≈ 0)


In [ ]:
"""Imports and global config — float64 throughout."""
import os, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.integrate import solve_ivp

DTYPE  = torch.float64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPS    = 1e-30
torch.set_default_dtype(DTYPE)
print(f"PyTorch {torch.__version__} | device={device} | dtype={DTYPE}")

PARAMS = {
    'ma'     : 100.0,
    'rho_m0' : 0.81,
    'rho_r0' : 2.7138e-4,
    'rho_L'  : 2.19,
    'a0'     : 1e-8,
    'phi0'   : 1.0,
    'dphi0'  : 0.0,
    't_init' : 1e-10,
    't_end'  : 1.0,
}

_LOG_T0 = np.log(PARAMS['t_init'])   # ln(1e-10) = -23.026
_LOG_T1 = np.log(PARAMS['t_end'])    # ln(1.0)   = 0
_LOG_DT = _LOG_T1 - _LOG_T0          # 23.026  (10 decades total)

# ── V14: sub-domain t ∈ [t_split_V14, t_end]  (τ_split = 0.20 → t ≈ 1e-8) ──
# Each 0.125 step in s = exactly 1 decade in t  (8 decades: 1e-8 → 1)
TAU_SPLIT    = 0.20
_LOG_T_SPLIT = _LOG_T0 + TAU_SPLIT * _LOG_DT    # = ln(1e-8) = -18.421
_SUB_LOG_DT  = _LOG_DT * (1.0 - TAU_SPLIT)      # = 0.80 × 23.026 = 18.421
T_SPLIT_NP   = float(np.exp(_LOG_T_SPLIT))       # = 1e-8

def t_to_s(t_tensor):
    """Map physical time t → normalised coordinate s ∈ [0,1]."""
    return (torch.log(torch.clamp(t_tensor, min=EPS)) - _LOG_T_SPLIT) / _SUB_LOG_DT

def s_to_t(s_tensor):
    """Inverse: s → t."""
    return torch.exp(_LOG_T_SPLIT + s_tensor * _SUB_LOG_DT)

print(f"PARAMS defined.")
print(f"t_split (V14) = {T_SPLIT_NP:.3e}  (tau_split={TAU_SPLIT})")
print(f"sub_log_dt    = {_SUB_LOG_DT:.4f}  (= 8 × ln10 exactly)")
print(f"s step per decade = {1.0/8.0:.4f}")
print(f"ds/dt @ t_split (1e-8) = {1.0/(T_SPLIT_NP*_SUB_LOG_DT):.2e}")
print(f"ds/dt @ t=1e-5         = {1.0/(1e-5*_SUB_LOG_DT):.1f}  (V13 boundary)")
print(f"ds/dt @ t=1e-3         = {1.0/(1e-3*_SUB_LOG_DT):.1f}")
print(f"ds/dt @ t=0.10         = {1.0/(0.10*_SUB_LOG_DT):.2f}")


In [ ]:
# ── ODE reference ─────────────────────────────────────────────────────────────
def _ode_rhs(t, y, ma, rho_m0, rho_r0, rho_L):
    a, phi, phi_dot = y
    a_s   = max(a, 1e-30)
    rax   = 0.5*phi_dot**2 + 0.5*ma**2*phi**2
    E1    = rax + rho_m0/a_s   + rho_r0/a_s**2 + rho_L*a**2
    da_dt = np.sqrt(max(E1, 0.0)/3.0)
    E2    = rax + rho_m0/a_s**3 + rho_r0/a_s**4 + rho_L
    H2    = np.sqrt(max(E2, 0.0)/3.0)
    return [da_dt, phi_dot, -np.sqrt(3.0)*H2*phi_dot - ma**2*phi]

def solve_ode(params, n_eval=5000):
    t0 = params['t_init']; t1 = params['t_end']
    y0 = [params['a0'], params['phi0'], params['dphi0']]
    args = (params['ma'], params['rho_m0'], params['rho_r0'], params['rho_L'])
    t_eval = np.logspace(np.log10(t0), np.log10(t1), n_eval)
    print("Solving ODE reference (RK45)...")
    t0w = time.time()
    sol = solve_ivp(_ode_rhs, (t0, t1), y0, t_eval=t_eval,
                    method='RK45', rtol=1e-10, atol=1e-13, args=args)
    print(f"  Done {time.time()-t0w:.2f}s | success={sol.success}")
    if not sol.success:
        warnings.warn(sol.message)
    return sol  # sol.y[0]=a(t), sol.y[1]=phi(t), sol.y[2]=dphi(t)


def physics_residuals(a, phi, dphi, da_dt, d_dphi_dt, ma, rho_m0, rho_r0, rho_L):
    """First-order residuals (no d2phi/dt2 required)."""
    sqrt3 = torch.tensor(3.0, dtype=DTYPE, device=a.device).sqrt()
    rax   = 0.5*dphi**2 + 0.5*ma**2*phi**2
    a_s   = torch.clamp(a, min=EPS)
    E1    = rax + rho_m0/a_s   + rho_r0/a_s**2 + rho_L*a**2
    H1    = torch.sqrt(torch.clamp(E1/3.0, min=0.0) + EPS)
    R_F   = da_dt - H1
    E2    = rax + rho_m0/a_s**3 + rho_r0/a_s**4 + rho_L
    H2    = torch.sqrt(torch.clamp(E2/3.0, min=0.0) + EPS)
    R_KG  = d_dphi_dt + sqrt3*H2*dphi + ma**2*phi
    return R_F, R_KG

print("ODE + physics residuals defined.")


In [ ]:
# ── Architecture ──────────────────────────────────────────────────────────────
class FCNN(nn.Module):
    def __init__(self, in_dim=1, out_dim=1, hidden=128, depth=4):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, out_dim))
        self.net = nn.Sequential(*layers)
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, s):
        return self.net(s)


class ScaleFactorNet(nn.Module):
    """a(t) = a_split * exp(clamp(s*g(s), max=60)),  a(t_split)=a_split exactly."""
    def __init__(self, a_split, hidden=128, depth=5):
        super().__init__()
        self.register_buffer('log_a_split', torch.tensor(np.log(float(a_split)), dtype=DTYPE))
        self.net = FCNN(1, 1, hidden, depth)

    def set_g_init(self, g_star):
        for m in reversed(list(self.net.net.modules())):
            if isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, float(g_star))
                break

    def forward(self, s):
        return torch.exp(self.log_a_split + torch.clamp(s * self.net(s), max=60.0))


class PhiNet(nn.Module):
    """WKB ansatz: phi = (A0 + s²·fA(s))·cos(ma·t) + (B0 + s²·fB(s))·sin(ma·t).
    Hard IC at s=0: phi(t_split) = A0·cos(...) + B0·sin(...) exactly."""
    def __init__(self, A0, B0, ma, hidden=128, depth=4):
        super().__init__()
        self.register_buffer('A0', torch.tensor(float(A0), dtype=DTYPE))
        self.register_buffer('B0', torch.tensor(float(B0), dtype=DTYPE))
        self.register_buffer('ma', torch.tensor(float(ma), dtype=DTYPE))
        self.net_A = FCNN(1, 1, hidden, depth)
        self.net_B = FCNN(1, 1, hidden, depth)

    def forward(self, s, t):
        A = self.A0 + s**2 * self.net_A(s)
        B = self.B0 + s**2 * self.net_B(s)
        return A * torch.cos(self.ma * t) + B * torch.sin(self.ma * t)


class DPhiNet(nn.Module):
    """WKB ansatz: dphi = (C0 + s²·fC(s))·cos(ma·t) + (D0 + s²·fD(s))·sin(ma·t).
    Hard IC at s=0: dphi(t_split) = C0·cos(...) + D0·sin(...) exactly."""
    def __init__(self, C0, D0, ma, hidden=128, depth=4):
        super().__init__()
        self.register_buffer('C0', torch.tensor(float(C0), dtype=DTYPE))
        self.register_buffer('D0', torch.tensor(float(D0), dtype=DTYPE))
        self.register_buffer('ma', torch.tensor(float(ma), dtype=DTYPE))
        self.net_C = FCNN(1, 1, hidden, depth)
        self.net_D = FCNN(1, 1, hidden, depth)

    def forward(self, s, t):
        C = self.C0 + s**2 * self.net_C(s)
        D = self.D0 + s**2 * self.net_D(s)
        return C * torch.cos(self.ma * t) + D * torch.sin(self.ma * t)


class AxionPINN_V14(nn.Module):
    """Three-network PINN restricted to s ∈ [0,1]  ↔  t ∈ [t_split_V14, t_end]."""
    def __init__(self, a_split, A0, B0, C0, D0, ma,
                 a_hidden=128, a_depth=5,
                 phi_hidden=128, phi_depth=4,
                 dphi_hidden=128, dphi_depth=4):
        super().__init__()
        self.register_buffer('_log_t_split', torch.tensor(_LOG_T_SPLIT, dtype=DTYPE))
        self.register_buffer('_sub_log_dt',  torch.tensor(_SUB_LOG_DT,  dtype=DTYPE))
        self.a_net    = ScaleFactorNet(a_split, a_hidden, a_depth)
        self.phi_net  = PhiNet(A0, B0, ma, phi_hidden, phi_depth)
        self.dphi_net = DPhiNet(C0, D0, ma, dphi_hidden, dphi_depth)

    def _to_s(self, t):
        return (torch.log(torch.clamp(t, min=EPS)) - self._log_t_split) / self._sub_log_dt

    def forward(self, t):
        s    = self._to_s(t)
        a    = self.a_net(s)
        phi  = self.phi_net(s, t)
        dphi = self.dphi_net(s, t)
        return a, phi, dphi


print("Classes defined: FCNN, ScaleFactorNet, PhiNet, DPhiNet, AxionPINN_V14")
n_params_estimate = (128*128*5 + 128*4*2) * 3  # rough estimate
print(f"Approx parameters per network (hidden=128, depth=5): ~{128*5*128 + 128:,}")


In [ ]:
import time as _time_mod

class AxionPINNSolver_V14:
    """
    V14: Extended sub-domain PINN covering t ∈ [t_split_V14=1e-8, t_end=1].

    Key design:
    * t_split_V14 = 1e-8  (tau_split=0.20, sub_log_dt=18.421)
    * s step of 0.125 = exactly 1 decade of physical time
    * WKB hard ICs at t_split from ODE (ma·t_split = 1e-6 rad, deeply frozen)
    * 8-stage curriculum (A→H) extending backward from s=0.875 to s=0.000
    * Stages A–E overlap with V13's proven region
    * Stages F–H (t < 1e-5): field completely frozen, physics trivial
    * Larger network capacity: hidden=128 (vs V13's 64) to cover 7 decades
    * For t < t_split_V14: ODE interpolation
    """

    def __init__(self, n_colloc=2000, hidden=128, depth_a=5, depth_phi=4, seed=42):
        torch.manual_seed(seed)
        np.random.seed(seed)

        # 1. ODE reference (full domain 1e-10 → 1)
        sol = solve_ode(PARAMS)
        self.t_ode    = sol.t
        self.a_ode    = sol.y[0]
        self.phi_ode  = sol.y[1]
        self.dphi_ode = sol.y[2]
        self.T_SPLIT  = float(np.exp(_LOG_T_SPLIT))   # = 1e-8

        # 2. ICs at t_split from ODE
        a_spl    = float(np.interp(self.T_SPLIT, self.t_ode, self.a_ode))
        phi_spl  = float(np.interp(self.T_SPLIT, self.t_ode, self.phi_ode))
        dphi_spl = float(np.interp(self.T_SPLIT, self.t_ode, self.dphi_ode))

        # 3. WKB rotation: exact IC recovery at s=0
        #    phi(t) = A·cos(ma·t) + B·sin(ma·t) → A,B from IC
        #    dphi(t) = C·cos(ma·t) + D·sin(ma·t) with C=ma·B, D=-ma·A (derivative)
        ma   = PARAMS['ma']
        c    = ma * self.T_SPLIT          # ma·t_split = 1e-6 rad (deeply frozen)
        A0   =  phi_spl * np.cos(c) - (dphi_spl / ma) * np.sin(c)
        B0   =  phi_spl * np.sin(c) + (dphi_spl / ma) * np.cos(c)
        C0   =  B0 * ma
        D0   = -A0 * ma
        print(f"t_split_V14 = {self.T_SPLIT:.3e} | ma*t_split = {c:.2e} rad (frozen)")
        print(f"A0={A0:.8f}  B0={B0:.8f}")
        print(f"C0={C0:.4e}  D0={D0:.4e}")
        print(f"IC check phi:  {A0*np.cos(c)+B0*np.sin(c):.10f}  (expect {phi_spl:.10f})")
        print(f"IC check dphi: {C0*np.cos(c)+D0*np.sin(c):.6e}   (expect {dphi_spl:.6e})")

        # 4. ODE-calibrated g_star for a_net (sub-domain t >= T_SPLIT only)
        mask_sub  = self.t_ode >= self.T_SPLIT * 0.9999
        s_sub     = (np.log(self.t_ode[mask_sub]) - _LOG_T_SPLIT) / _SUB_LOG_DT
        log_a_rel = np.log(self.a_ode[mask_sub] / a_spl)
        g_star    = float(np.dot(s_sub, log_a_rel) / np.dot(s_sub, s_sub))
        print(f"g_star (V14 sub-domain) = {g_star:.4f}")

        # 5. Scales (over sub-domain only)
        self.phi_scale  = float(max(np.max(np.abs(self.phi_ode)), 1e-4))
        self.dphi_scale = float(max(np.max(np.abs(self.dphi_ode[mask_sub])), 1e-4))
        print(f"phi_scale={self.phi_scale:.2f}  dphi_scale={self.dphi_scale:.3e}")

        # 6. Reference tensors (sub-domain only, for anchor loss and pretrain)
        t_sub = self.t_ode[mask_sub]
        self.t_ref_torch    = torch.tensor(t_sub[:, None],                     dtype=DTYPE)
        self.la_ref         = torch.tensor(np.log(self.a_ode[mask_sub, None]), dtype=DTYPE)
        self.phi_ref_torch  = torch.tensor(self.phi_ode[mask_sub, None],       dtype=DTYPE)
        self.dphi_ref_torch = torch.tensor(self.dphi_ode[mask_sub, None],      dtype=DTYPE)

        # 7. Build model
        self.model = AxionPINN_V14(
            a_split=a_spl, A0=A0, B0=B0, C0=C0, D0=D0, ma=ma,
            a_hidden=hidden, a_depth=depth_a,
            phi_hidden=hidden, phi_depth=depth_phi,
            dphi_hidden=hidden, dphi_depth=depth_phi,
        )
        self.model.a_net.set_g_init(g_star)
        self.n_colloc = n_colloc
        self.loss_log = []

    # ── collocation with curriculum support ────────────────────────────────────
    def _make_colloc(self, n=None, s_min=0.0):
        """Sample collocation points uniformly in s ∈ [s_min, 1]."""
        if n is None:
            n = self.n_colloc
        s_b = s_min + (1.0 - s_min) * torch.rand(n, 1, dtype=DTYPE)
        return s_to_t(s_b).requires_grad_(True)

    # ── physics loss ──────────────────────────────────────────────────────────
    def _physics_loss(self, t_col):
        p = PARAMS
        a, phi, dphi = self.model(t_col)
        ones      = torch.ones_like(a)
        da_dt     = torch.autograd.grad(a,    t_col, grad_outputs=ones,
                                        create_graph=True, retain_graph=True)[0]
        d_dphi_dt = torch.autograd.grad(dphi, t_col, grad_outputs=ones,
                                        create_graph=True, retain_graph=True)[0]
        d_phi_dt  = torch.autograd.grad(phi,  t_col, grad_outputs=ones,
                                        create_graph=True)[0]
        R_F, R_KG = physics_residuals(
            a, phi, dphi, da_dt, d_dphi_dt,
            p['ma'], p['rho_m0'], p['rho_r0'], p['rho_L'])
        R_phi = d_phi_dt - dphi
        with torch.no_grad():
            a_d   = a.detach().clamp(min=EPS)
            rax_d = 0.5*dphi.detach()**2 + 0.5*p['ma']**2*phi.detach()**2
            E1_d  = rax_d + p['rho_m0']/a_d + p['rho_r0']/a_d**2 + p['rho_L']*a_d**2
            H_ref = torch.sqrt(torch.clamp(E1_d/3.0, min=1e-60))
        l_F   = torch.mean((R_F   / (H_ref + EPS))**2)
        l_KG  = torch.mean((R_KG  / (p['ma']**2 * self.phi_scale))**2)
        l_phi = torch.mean((R_phi / (self.dphi_scale + EPS))**2)
        return l_F, l_KG, l_phi

    # ── anchor loss ───────────────────────────────────────────────────────────
    def _anchor_loss(self):
        a_p, phi_p, dphi_p = self.model(self.t_ref_torch)
        la_p = torch.log(torch.clamp(a_p, min=EPS))
        l_a    = torch.mean((la_p   - self.la_ref)**2)
        l_phi  = torch.mean((phi_p  - self.phi_ref_torch)**2)  / self.phi_scale**2
        l_dphi = torch.mean((dphi_p - self.dphi_ref_torch)**2) / self.dphi_scale**2
        return l_a + l_phi + l_dphi

    # ── pretrain on ODE reference ─────────────────────────────────────────────
    def pretrain_a(self, epochs=8000, lr=1e-3, print_every=2000):
        opt = torch.optim.Adam(self.model.a_net.parameters(), lr=lr)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr*0.01)
        print(f"=== Pretrain a_net ({epochs} ep) ===")
        for ep in range(1, epochs+1):
            opt.zero_grad()
            s_v  = self.model._to_s(self.t_ref_torch)
            a_p  = self.model.a_net(s_v)
            loss = torch.mean((torch.log(torch.clamp(a_p, min=EPS)) - self.la_ref)**2)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.a_net.parameters(), 1.0)
            opt.step(); sch.step()
            if ep % print_every == 0:
                print(f"  ep {ep:5d} | loss={loss.item():.3e}")
        with torch.no_grad():
            s_v = self.model._to_s(self.t_ref_torch)
            re  = (torch.exp(self.la_ref) - self.model.a_net(s_v)).abs() / \
                  torch.exp(self.la_ref).abs()
        print(f"  Done -- median {re.median().item():.3e}  max {re.max().item():.3e}")

    def pretrain_phi(self, epochs=5000, lr=1e-3, print_every=1000):
        opt = torch.optim.Adam(self.model.phi_net.parameters(), lr=lr)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr*0.01)
        print(f"=== Pretrain phi_net ({epochs} ep, WKB) ===")
        for ep in range(1, epochs+1):
            opt.zero_grad()
            s_v   = self.model._to_s(self.t_ref_torch)
            phi_p = self.model.phi_net(s_v, self.t_ref_torch)
            loss  = torch.mean((phi_p - self.phi_ref_torch)**2) / self.phi_scale**2
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.phi_net.parameters(), 1.0)
            opt.step(); sch.step()
            if ep % print_every == 0:
                print(f"  ep {ep:5d} | loss={loss.item():.3e}")
        with torch.no_grad():
            s_v = self.model._to_s(self.t_ref_torch)
            re  = (self.phi_ref_torch - self.model.phi_net(s_v, self.t_ref_torch)).abs() / \
                  (self.phi_ref_torch.abs() + 1e-30)
        print(f"  Done -- median {re.median().item():.3e}  max {re.max().item():.3e}")

    def pretrain_dphi(self, epochs=5000, lr=1e-3, print_every=1000):
        opt = torch.optim.Adam(self.model.dphi_net.parameters(), lr=lr)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr*0.01)
        print(f"=== Pretrain dphi_net ({epochs} ep, WKB) ===")
        for ep in range(1, epochs+1):
            opt.zero_grad()
            s_v    = self.model._to_s(self.t_ref_torch)
            dphi_p = self.model.dphi_net(s_v, self.t_ref_torch)
            loss   = torch.mean((dphi_p - self.dphi_ref_torch)**2) / self.dphi_scale**2
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.dphi_net.parameters(), 1.0)
            opt.step(); sch.step()
            if ep % print_every == 0:
                print(f"  ep {ep:5d} | loss={loss.item():.3e}")
        with torch.no_grad():
            s_v = self.model._to_s(self.t_ref_torch)
            re  = (self.dphi_ref_torch - self.model.dphi_net(s_v, self.t_ref_torch)).abs() / \
                  (self.dphi_ref_torch.abs() + 1e-30)
        print(f"  Done -- median {re.median().item():.3e}  max {re.max().item():.3e}")

    # ── Adam curriculum stage ─────────────────────────────────────────────────
    def _train_stage(self, label, epochs, lr, s_min,
                     lam_kg=10.0, lam_phi=0.1, lam_anc=0.1, print_every=None):
        opt   = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=1e-6)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr*0.01)
        if print_every is None:
            print_every = max(1, epochs // 20)
        t_min_phys   = float(s_to_t(torch.tensor([[s_min]])).item())
        dsdt_at_smin = 1.0 / (t_min_phys * _SUB_LOG_DT) if t_min_phys > 0 else float('inf')
        print(f"\n=== {label}  (s>={s_min:.3f}, t>={t_min_phys:.1e}, "
              f"ds/dt_left={dsdt_at_smin:.2e}, {epochs} ep, lr={lr:.1e}) ===")
        t0 = _time_mod.time()
        for ep in range(1, epochs+1):
            opt.zero_grad()
            t_col = self._make_colloc(s_min=s_min)
            l_F, l_KG, l_phi = self._physics_loss(t_col)
            l_anc = self._anchor_loss()
            loss  = l_F + lam_kg*l_KG + lam_phi*l_phi + lam_anc*l_anc
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            opt.step(); sched.step()
            self.loss_log.append({
                'ep': ep, 'label': label, 's_min': s_min,
                'F': l_F.item(), 'KG': l_KG.item(),
                'phi_cons': l_phi.item(), 'anc': l_anc.item(),
                'total': loss.item()
            })
            if ep % print_every == 0:
                print(f"  ep {ep:5d}/{epochs}  F={l_F.item():.2e}  "
                      f"KG={l_KG.item():.2e}  phi_cons={l_phi.item():.2e}  "
                      f"anc={l_anc.item():.2e}  t={_time_mod.time()-t0:.0f}s")

    # ── L-BFGS polish ──────────────────────────────────────────────────────────
    def lbfgs_polish(self, max_iter=500, s_min=0.0,
                     lam_kg=10.0, lam_phi=1.0, lam_anc=0.1):
        print(f"\n=== L-BFGS Polish ({max_iter} iter, s>={s_min:.3f}) ===")
        opt   = torch.optim.LBFGS(self.model.parameters(), max_iter=max_iter,
                                   tolerance_grad=1e-9, history_size=50,
                                   line_search_fn='strong_wolfe')
        t_col = self._make_colloc(n=4000, s_min=s_min)
        _last = {}
        def closure():
            opt.zero_grad()
            l_F, l_KG, l_phi = self._physics_loss(t_col)
            l_anc = self._anchor_loss()
            loss  = l_F + lam_kg*l_KG + lam_phi*l_phi + lam_anc*l_anc
            loss.backward()
            _last.update({'F': l_F.item(), 'KG': l_KG.item(),
                          'phi': l_phi.item(), 'anc': l_anc.item()})
            return loss
        opt.step(closure)
        print(f"  Final: F={_last.get('F',float('nan')):.2e}  "
              f"KG={_last.get('KG',float('nan')):.2e}  "
              f"phi_cons={_last.get('phi',float('nan')):.2e}  "
              f"anc={_last.get('anc',float('nan')):.2e}")

    # ── Full 8-stage curriculum ────────────────────────────────────────────────
    def train_all(self):
        t_wall = _time_mod.time()
        print("=" * 70)
        print("V14 Training  --  t ∈ [1e-8, 1]  (8-stage curriculum, hidden=128)")
        print("=" * 70)
        print("Stage summary (s step=0.125 = 1 decade per stage):")
        for s_min, t_min, dsdt in [
            (0.875,'1e-1',0.54), (0.750,'1e-2',5.4), (0.625,'1e-3',54),
            (0.500,'1e-4',543),  (0.375,'1e-5',5430),(0.250,'1e-6',54300),
            (0.125,'1e-7',543000),(0.000,'1e-8',5430000)]:
            print(f"  s>={s_min:.3f}  t>={t_min}  ds/dt~{dsdt:.2e}")

        # Pre-training on ODE reference
        print("\n[1/9] Pre-training (ODE supervised)")
        self.pretrain_a(epochs=8000,  lr=1e-3, print_every=2000)
        self.pretrain_phi(epochs=5000,  lr=1e-3, print_every=1000)
        self.pretrain_dphi(epochs=5000, lr=1e-3, print_every=1000)

        # Stage A: s >= 0.875  (t >= 0.10, late-time oscillations)
        self._train_stage("Stage A (s>=0.875, t>=1e-1)", epochs=4000, lr=5e-4,
                          s_min=0.875, lam_kg=10.0, lam_phi=0.1, lam_anc=0.5)

        # Stage B: s >= 0.750  (t >= 0.01, field oscillating rapidly)
        self._train_stage("Stage B (s>=0.750, t>=1e-2)", epochs=5000, lr=3e-4,
                          s_min=0.750, lam_kg=10.0, lam_phi=0.1, lam_anc=0.3)

        # Stage C: s >= 0.625  (t >= 1e-3, oscillation onset)
        self._train_stage("Stage C (s>=0.625, t>=1e-3)", epochs=6000, lr=2e-4,
                          s_min=0.625, lam_kg=10.0, lam_phi=0.1, lam_anc=0.2)

        # Stage D: s >= 0.500  (t >= 1e-4)
        self._train_stage("Stage D (s>=0.500, t>=1e-4)", epochs=7000, lr=1e-4,
                          s_min=0.500, lam_kg=10.0, lam_phi=0.1, lam_anc=0.2)

        # Stage E: s >= 0.375  (t >= 1e-5, V13 split boundary)
        self._train_stage("Stage E (s>=0.375, t>=1e-5)", epochs=8000, lr=1e-4,
                          s_min=0.375, lam_kg=10.0, lam_phi=0.1, lam_anc=0.2)

        # Stage F: s >= 0.250  (t >= 1e-6, new V14 territory)
        self._train_stage("Stage F (s>=0.250, t>=1e-6)", epochs=8000, lr=5e-5,
                          s_min=0.250, lam_kg=10.0, lam_phi=0.1, lam_anc=0.15)

        # Stage G: s >= 0.125  (t >= 1e-7, deeply frozen)
        self._train_stage("Stage G (s>=0.125, t>=1e-7)", epochs=10000, lr=3e-5,
                          s_min=0.125, lam_kg=10.0, lam_phi=0.1, lam_anc=0.1)

        # Stage H: s >= 0.000  (t >= 1e-8, full V14 domain)
        self._train_stage("Stage H (s>=0.000, t>=1e-8)", epochs=12000, lr=1e-5,
                          s_min=0.000, lam_kg=10.0, lam_phi=0.1, lam_anc=0.1)

        # L-BFGS polish on full V14 domain
        self.lbfgs_polish(max_iter=500, s_min=0.000)

        total_min = (_time_mod.time() - t_wall) / 60.0
        print(f"\nTotal wall time: {total_min:.1f} min")

    # ── Hybrid evaluation ─────────────────────────────────────────────────────
    def evaluate(self, n_eval=1000):
        """Hybrid: ODE for t < t_split_V14, PINN for t >= t_split_V14."""
        t_eval    = np.logspace(np.log10(PARAMS['t_init']),
                                np.log10(PARAMS['t_end']), n_eval)
        a_pred    = np.empty_like(t_eval)
        phi_pred  = np.empty_like(t_eval)
        dphi_pred = np.empty_like(t_eval)
        mask_ode  = t_eval <  self.T_SPLIT
        mask_pinn = t_eval >= self.T_SPLIT
        if mask_ode.any():
            a_pred[mask_ode]    = np.interp(t_eval[mask_ode], self.t_ode, self.a_ode)
            phi_pred[mask_ode]  = np.interp(t_eval[mask_ode], self.t_ode, self.phi_ode)
            dphi_pred[mask_ode] = np.interp(t_eval[mask_ode], self.t_ode, self.dphi_ode)
        if mask_pinn.any():
            t_ptorch = torch.tensor(t_eval[mask_pinn, None], dtype=DTYPE)
            with torch.no_grad():
                ap, pp, dp = self.model(t_ptorch)
            a_pred[mask_pinn]    = ap.numpy().ravel()
            phi_pred[mask_pinn]  = pp.numpy().ravel()
            dphi_pred[mask_pinn] = dp.numpy().ravel()
        return t_eval, a_pred, phi_pred, dphi_pred


print("AxionPINNSolver_V14 defined.")
print(f"Total expected Adam epochs: {8000+5000+5000+4000+5000+6000+7000+8000+8000+10000+12000:,} + L-BFGS 500")


In [ ]:
def plot_comparison_v14(solver, save_dir='results_v14'):
    os.makedirs(save_dir, exist_ok=True)
    t_eval, a_pred, phi_pred, dphi_pred = solver.evaluate()
    t_ode, a_ode, phi_ode, dphi_ode = solver.t_ode, solver.a_ode, solver.phi_ode, solver.dphi_ode
    T_SPLIT = solver.T_SPLIT
    mask_pinn = t_eval >= T_SPLIT

    a_ref    = np.interp(t_eval, t_ode, a_ode)
    phi_ref  = np.interp(t_eval, t_ode, phi_ode)
    dphi_ref = np.interp(t_eval, t_ode, dphi_ode)
    rel_a    = np.abs(a_pred    - a_ref)    / (np.abs(a_ref)    + 1e-30)
    rel_phi  = np.abs(phi_pred  - phi_ref)  / (np.abs(phi_ref)  + 1e-30)
    rel_dphi = np.abs(dphi_pred - dphi_ref) / (np.abs(dphi_ref) + 1e-6)

    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    fig.suptitle(f"PINN V14 vs ODE  |  PINN region: t >= {T_SPLIT:.0e}  (7 decades)", fontsize=12)

    for ax, yt, yp, lab in zip(
            [axes[0,0], axes[1,0], axes[2,0]],
            [a_ode, phi_ode, dphi_ode],
            [a_pred, phi_pred, dphi_pred],
            ['a(t)', 'phi(t)', 'dphi(t)']):
        ax.semilogx(t_ode, yt, 'k-', lw=1.5, label='ODE')
        ax.semilogx(t_eval[mask_pinn], yp[mask_pinn], 'r--', lw=1.5, label='PINN V14')
        ax.axvline(T_SPLIT, color='blue',  lw=0.8, ls=':', label=f't_split={T_SPLIT:.0e}')
        ax.axvline(1e-5,    color='green', lw=0.8, ls=':', label='V13 boundary (1e-5)')
        ax.set_xlabel('t'); ax.set_ylabel(lab)
        ax.set_title(lab); ax.legend(fontsize=7); ax.grid(True, which='both', alpha=0.2)

    for ax, rel, lab in zip(
            [axes[0,1], axes[1,1], axes[2,1]],
            [rel_a, rel_phi, rel_dphi],
            ['|da|/a', '|dphi|/|phi|', '|ddphi|/|dphi|']):
        ax.loglog(t_eval[mask_pinn], rel[mask_pinn] + 1e-16, 'r-', lw=1.2)
        ax.axhline(1e-2, color='orange', ls='--', lw=0.8, label='1%')
        ax.axhline(1e-4, color='green',  ls='--', lw=0.8, label='0.01%')
        ax.set_xlabel('t'); ax.set_ylabel(lab); ax.set_title(f'Relative error {lab}')
        ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.2)

    plt.tight_layout()
    path = os.path.join(save_dir, 'v14_comparison.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {path}")

    with np.errstate(invalid='ignore'):
        print(f"\nPINN region (t >= {T_SPLIT:.0e}) summary:")
        print(f"  a(t)   median={np.median(rel_a[mask_pinn]):.2e}  90th={np.percentile(rel_a[mask_pinn],90):.2e}")
        phi_fin = rel_phi[mask_pinn][np.isfinite(rel_phi[mask_pinn])]
        print(f"  phi(t) 90th  ={np.percentile(phi_fin,90):.2e}")
        dphi_fin = rel_dphi[mask_pinn][np.isfinite(rel_dphi[mask_pinn])]
        print(f"  dphi   90th  ={np.percentile(dphi_fin,90):.2e}")


def plot_loss_v14(solver, save_dir='results_v14'):
    os.makedirs(save_dir, exist_ok=True)
    if not solver.loss_log:
        print("No loss history."); return
    fig, ax = plt.subplots(figsize=(16, 5))
    ep_all   = [d['ep']       for d in solver.loss_log]
    F_all    = [d['F']        for d in solver.loss_log]
    KG_all   = [d['KG']       for d in solver.loss_log]
    phi_all  = [d['phi_cons'] for d in solver.loss_log]
    ax.semilogy(ep_all, np.array(F_all)   + 1e-20, label='Friedmann', lw=1, alpha=0.8)
    ax.semilogy(ep_all, np.array(KG_all)  + 1e-20, label='KG',        lw=1, alpha=0.8)
    ax.semilogy(ep_all, np.array(phi_all) + 1e-20, label='phi_cons',  lw=1, alpha=0.8)
    ax.set_xlabel('Epoch (per stage)'); ax.set_ylabel('Loss')
    ax.set_title('V14 Loss History (8-stage curriculum, t=[1e-8,1])'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'v14_loss.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.show()
    print(f"Saved: {path}")


def region_summary_v14(solver):
    """Per-decade accuracy table comparing V13-covered vs V14-new regions."""
    t_eval, a_pred, phi_pred, _ = solver.evaluate(n_eval=2000)
    t_ode, a_ode, phi_ode = solver.t_ode, solver.a_ode, solver.phi_ode
    a_ref   = np.interp(t_eval, t_ode, a_ode)
    phi_ref = np.interp(t_eval, t_ode, phi_ode)
    rel_a   = np.abs(a_pred   - a_ref)   / (np.abs(a_ref)   + 1e-30)
    rel_phi = np.abs(phi_pred - phi_ref) / (np.abs(phi_ref) + 1e-30)

    decades = [(1e-8, 1e-7, 'V14-new [1e-8, 1e-7]'),
               (1e-7, 1e-6, 'V14-new [1e-7, 1e-6]'),
               (1e-6, 1e-5, 'V14-new [1e-6, 1e-5]'),
               (1e-5, 1e-4, 'V13:    [1e-5, 1e-4]'),
               (1e-4, 1e-3, 'V13:    [1e-4, 1e-3]'),
               (1e-3, 1e-2, 'V13:    [1e-3, 1e-2]'),
               (1e-2, 1e-1, '        [1e-2, 1e-1]'),
               (1e-1, 1.0,  '        [1e-1, 1.0 ]')]
    print("\n┌──────────────────────────────┬─────────────┬─────────────┐")
    print("│ Region                       │  a(t) 90th% │ phi(t) 90th%│")
    print("├──────────────────────────────┼─────────────┼─────────────┤")
    for t_lo, t_hi, label in decades:
        mask = (t_eval >= t_lo) & (t_eval < t_hi) & (t_eval >= solver.T_SPLIT)
        if mask.sum() < 3:
            print(f"│ {label:<28}  │     ---     │     ---     │")
            continue
        a90   = np.percentile(rel_a[mask], 90)
        phi90 = np.percentile(rel_phi[mask][np.isfinite(rel_phi[mask])], 90) \
                if np.isfinite(rel_phi[mask]).any() else float('nan')
        print(f"│ {label:<28}  │  {a90:.2e}   │  {phi90:.2e}   │")
    print("└──────────────────────────────┴─────────────┴─────────────┘")


print("Plotting utilities (V14) defined.")


In [ ]:
# ── SMOKE TEST ──────────────────────────────────────────────────────────────
print("=" * 65)
print("SMOKE TEST — V14  (t_split=1e-8, sub_log_dt=18.421, hidden=128)")
print("=" * 65)

torch.manual_seed(42)
solver_v14 = AxionPINNSolver_V14(n_colloc=2000, hidden=128, seed=42)

# Untrained full-domain losses
t_full = solver_v14._make_colloc(n=4000, s_min=0.0)
lF, lKG, lP = solver_v14._physics_loss(t_full)
lA = solver_v14._anchor_loss()
print(f"\nUntrained losses (full s in [0,1], t in [1e-8, 1]):")
print(f"  Friedmann  = {lF:.3e}")
print(f"  KG         = {lKG:.3e}")
print(f"  phi_cons   = {lP:.3e}   (WKB IC → expect < 0.10)")
print(f"  Anchor     = {lA:.3e}")

# Per-stage losses
print("\nPer-curriculum-stage (untrained):")
stages = [(0.875,'A t>=0.1'), (0.750,'B t>=1e-2'), (0.625,'C t>=1e-3'),
          (0.500,'D t>=1e-4'), (0.375,'E t>=1e-5'), (0.250,'F t>=1e-6'),
          (0.125,'G t>=1e-7'), (0.000,'H t>=1e-8')]
for s_min, label in stages:
    t_c = solver_v14._make_colloc(n=800, s_min=s_min)
    lF2, lKG2, lP2 = solver_v14._physics_loss(t_c)
    t_min_np = float(s_to_t(torch.tensor([[s_min]], dtype=DTYPE)))
    ds_dt    = 1.0 / (t_min_np * solver_v14.model._sub_log_dt.item())
    print(f"  Stage {label}:  F={lF2:.2e}  KG={lKG2:.2e}  phi_cons={lP2:.2e}  ds/dt={ds_dt:.2e}")

# Forward pass check
t_check = torch.tensor([[1e-5]], dtype=DTYPE)
a_out, phi_out, dphi_out = solver_v14.model(t_check)
print(f"\nForward pass at t=1e-5: a={float(a_out):.6f}, phi={float(phi_out):.6f}, dphi={float(dphi_out):.4e}")
t_check2 = torch.tensor([[solver_v14.T_SPLIT]], dtype=DTYPE)
a_out2, phi_out2, dphi_out2 = solver_v14.model(t_check2)
print(f"Forward pass at t_split=1e-8: a={float(a_out2):.2e}, phi={float(phi_out2):.8f}, dphi={float(dphi_out2):.4e}")
nan_check = any(np.isnan(v) for v in [float(a_out), float(phi_out), float(dphi_out),
                                       float(a_out2), float(phi_out2), float(dphi_out2)])
print("Smoke test PASSED ✓" if not nan_check else "FAILED — NaN detected!")


In [ ]:
# ── TRAINING ────────────────────────────────────────────────────────────────
import os

RESULTS_DIR_V14 = os.path.join(os.getcwd(), 'results_v14')
os.makedirs(RESULTS_DIR_V14, exist_ok=True)

print("Starting V14 training — 8-stage curriculum (t_split=1e-8, hidden=128)")
print(f"Expected total: ~78,000 Adam epochs + L-BFGS 500 steps")
print("-" * 65)

t_start_train = _time_mod.time()
solver_v14.train_all()
t_elapsed = _time_mod.time() - t_start_train

CKPT_PATH_V14 = os.path.join(RESULTS_DIR_V14, 'v14_model.pt')
torch.save({
    'model_state': solver_v14.model.state_dict(),
    'loss_log':    solver_v14.loss_log,
    'T_SPLIT':     solver_v14.T_SPLIT,
    'params':      PARAMS,
}, CKPT_PATH_V14)

print(f"\nTraining complete in {t_elapsed/60:.1f} min")
print(f"Checkpoint saved → {CKPT_PATH_V14}")
if solver_v14.loss_log:
    last = solver_v14.loss_log[-1]
    print(f"Final losses (last record):  F={last['F']:.2e}  KG={last['KG']:.2e}  phi_cons={last['phi_cons']:.2e}")


In [ ]:
# ── EVALUATION & PLOTS ──────────────────────────────────────────────────────
plot_loss_v14(solver_v14, save_dir=RESULTS_DIR_V14)
plot_comparison_v14(solver_v14, save_dir=RESULTS_DIR_V14)
region_summary_v14(solver_v14)

# ── Full domain overview plot ────────────────────────────────────────────────
t_eval, a_pred, phi_pred, _ = solver_v14.evaluate(n_eval=2000)
t_ode, a_ode, phi_ode = solver_v14.t_ode, solver_v14.a_ode, solver_v14.phi_ode

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for ax, yp, yo, lab in zip(axes, [a_pred, phi_pred], [a_ode, phi_ode], ['a(t)', 'phi(t)']):
    ax.semilogx(t_ode, yo, 'k-', lw=2, label='ODE reference')
    mask_all = t_eval >= solver_v14.T_SPLIT
    ax.semilogx(t_eval[mask_all], yp[mask_all], 'r--', lw=1.5, label='PINN V14')
    ax.axvline(solver_v14.T_SPLIT, color='purple', ls=':', lw=1.0, label=f't_split={solver_v14.T_SPLIT:.0e} (V14)')
    ax.axvline(1e-5, color='green',  ls=':', lw=1.0, label='V13 boundary (1e-5)')
    ax.axvline(1e-3, color='blue',   ls=':', lw=1.0, label='V12 boundary (1e-3)')
    ax.set_ylabel(lab); ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.2)
    ax.set_xlim([solver_v14.T_SPLIT * 0.9, 1.1])
axes[-1].set_xlabel('t')
fig.suptitle('V14: PINN covers t ∈ [1e-8, 1]  (7 decades vs V12: 3, V13: 5)', fontsize=12)
plt.tight_layout()
path_full = os.path.join(RESULTS_DIR_V14, 'v14_full_domain.png')
plt.savefig(path_full, dpi=150, bbox_inches='tight'); plt.show()
print(f"Saved: {path_full}")
